# ICT-50 -- Origami Wolfram pli 2 : la regle 30 comme generateur d'alea, la regle 110 et son universalite

**Navigation** : [Index](README.md) | Precedent dans la serie : [ICT-49 -- Machines de Godel](ICT-49-MachinesDeGoedel-Python.ipynb)

**Carnet** : deux points de la definition du pli 2 qu'aucun carnet de la serie ne couvre encore --
la **regle 30 comme generateur de sequences aleatoires** et l'**universalite de la regle 110** --
avec, a cote de chaque these, sa **reception critique**.

**Organe utilise** : `ict.wolfram_step` (livre par PR #19793, chemin
`MyIA.AI.Notebooks/IIT/ICT-Series/ict/wolfram_step.py`). Toutes les mesures de ce carnet passent
par `wolfram_step` ; **aucune reimplementation locale de l'automate**. Ce que le carnet ecrit
lui-meme, ce sont les **instruments de mesure** (tests statistiques, complexite lineaire), qui ne
sont pas dans l'organe.

**Ce que ce carnet fait, et ce qu'il ne fait pas.**

| Ces deux theses | Statut | Ce que fait ce carnet |
|---|---|---|
| La regle 30 produit des sequences utilisables comme alea | *these empirique* de Wolfram 1986 | il **mesure** la sequence et la confronte a une batterie de tests standard, avec temoins negatifs |
| La regle 110 est universelle (Turing-complete) | *theoreme* (Cook 2004) | il **ne le re-prouve pas** ; il mesure la structure d'ether qui porte la construction, et expose la reception |

La distinction est le sujet d'une partie de ce carnet : une these mesuree et un theoreme ne se
citent pas de la meme facon.

## Sources primaires et regle de citation

Les quatre textes utilises sont lus dans les copies archivees du gisement
`G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\` :

| Reference | Copie archivee |
|---|---|
| Wolfram, *Random Sequence Generation by Cellular Automata*, Adv. Appl. Math. 7 (1986) | `1986 - Wolfram - Random Sequence Generation by Cellular Automata (Adv. Appl. Math. 7, 1986).pdf` |
| Wolfram, *Universality and Complexity in Cellular Automata*, Physica D 10 (1984) | `1984 - Wolfram - Universality and Complexity in Cellular Automata (Physica D 10, 1984).pdf` |
| Cook, *Universality in Elementary Cellular Automata*, Complex Systems 15 (2004) | `2004 - Cook - Universality in Elementary Cellular Automata (Complex Systems 15-1, OA).pdf` |
| Gray, *A Mathematician Looks at Wolfram's New Kind of Science*, Notices AMS 50-2 (2003) | `2002 - Gray - A Mathematician Looks at Wolfram's New Kind of Science (Notices AMS 50-2 2003, OA).pdf` |

**Convention de citation** : chaque affirmation attribuee a un texte cite la **page de la copie
archivee** et la section. Une capture HTML se citerait par son **titre de section**, jamais par un
numero de ligne -- le numero de ligne d'un texte extrait ne correspond a rien dans le fichier.

In [1]:
import sys
import numpy as np
from scipy import stats

sys.path.insert(0, 'MyIA.AI.Notebooks/IIT/ICT-Series')
from ict.wolfram_step import wolfram_step, classify_rule, CANONICAL_RULES

print('ict.wolfram_step importe OK -- aucune reimplementation locale de l automate')
print('CANONICAL_RULES:', CANONICAL_RULES)

ict.wolfram_step importe OK -- aucune reimplementation locale de l automate
CANONICAL_RULES: (0, 30, 110, 250)


## 1. Le protocole de Wolfram : extraire une sequence d'un registre circulaire

Wolfram 1986 section 11 (p. 162) donne le mode d'emploi de l'automate comme generateur :

> Random sequences can be obtained by sampling the sequence of values of a particular site in a
> register updated according to Eq. (3.1).

Les parametres viennent de la legende de la Table 10.1 (p. 159) : registre **circulaire** de
longueur `N`, **graine consistant en un seul site non nul**, echantillon de longueur `L`.

Deux details du protocole ne sont **pas** donnes par le texte : **quel site** on echantillonne, et
**ou** se trouve le site non nul dans le registre. Le carnet les fixe explicitement, et il teste
plus bas la sensibilite du resultat a ce choix -- c'est la premiere chose a verifier quand une
reproduction ne tombe pas sur les chiffres publies.

In [2]:
def seed_single_nonzero(n_cells, seed_site=None):
    """Graine de la Table 10.1 : un seul site non nul, au centre par defaut."""
    state = np.zeros(n_cells, dtype=int)
    state[n_cells // 2 if seed_site is None else seed_site] = 1
    return state


def temporal_bits(rule, n_cells, n_bits, site=0, seed_site=None):
    """Valeurs d'UN site, un echantillon par pas de temps (Wolfram 1986, sec. 11)."""
    state = seed_single_nonzero(n_cells, seed_site)
    out = np.empty(n_bits, dtype=np.uint8)
    for t in range(n_bits):
        out[t] = state[site]
        state = wolfram_step(state, rule)
    return out


def bits_to_bytes(bits):
    """Regroupe en octets non chevauchants (Annexe A : blocs de longueur 8)."""
    n = (len(bits) // 8) * 8
    return np.packbits(bits[:n].astype(np.uint8))


bits = temporal_bits(30, 49, 64)
print('64 premiers bits du site 0, regle 30, N=49 :', ''.join(map(str, bits)))

64 premiers bits du site 0, regle 30, N=49 : 0000000000000000000000001011100011010011000101011110110110111010


## 2. La mise en garde que Wolfram ecrit lui-meme : periode contre longueur d'echantillon

La legende de la Table 10.1 (p. 159) porte une reserve que le corps du texte repete :

> Accumulations close to 0 or 1 suggest deviations from randomness. Such accumulations are seen in
> this case only when the period of the cellular automaton is comparable to the length of the
> sequence sampled.

Un registre **fini** est periodique : au-dela de sa periode, la sequence qu'on en extrait n'est plus
un echantillon de comportement, c'est une repetition. Wolfram publie les periodes en Table 9.2 mais
ne les reprend pas dans la Table 10.1. Mesurons-les ici, pour chaque taille utilisee, et comparons
au rapport `L / periode`.

In [3]:
def period_of(rule, n_cells, max_steps=300000):
    """Periode exacte du registre circulaire (et pre-periode), par detection d'etat repete."""
    state = seed_single_nonzero(n_cells)
    seen, t = {}, 0
    while t < max_steps:
        key = state.tobytes()
        if key in seen:
            return t - seen[key], seen[key]
        seen[key] = t
        state = wolfram_step(state, rule)
        t += 1
    return None, None


L = 64 * 1024
periodes = {}
for N in (17, 23, 29, 37, 49):
    p, pre = period_of(30, N)
    periodes[N] = p
    ratio = f'{L / p:.2f}' if p else 'non mesuree (>300k pas)'
    print(f'N={N:3d}  periode={p}  pre-periode={pre}  L/periode={ratio}')

N= 17  periode=10846  pre-periode=56  L/periode=6.04


N= 23  periode=38249  pre-periode=1161  L/periode=1.71


N= 29  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


N= 37  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


N= 49  periode=None  pre-periode=None  L/periode=non mesuree (>300k pas)


**Lecture du resultat** : a `N=17` la periode vaut 10 846 pas pour un echantillon de 65 536 bits --
le registre fait **six fois le tour** de son cycle pendant la mesure, et la sequence extraite n'est
pas 65 536 pas de dynamique mais la repetition d'un motif de 10 846. A `N=23` le rapport tombe a
1,71 : le registre fait encore plus d'un tour. A partir de `N=29` la periode depasse la fenetre de
recherche (300 000 pas) et le rapport devient tres inferieur a 1.

C'est exactement la condition que Wolfram designe. Elle a une consequence methodologique : un test
qui rejette la regle 30 a `N=17` ne dit **rien** sur le caractere aleatoire de la regle 30 -- il
detecte la periodicite du **registre**, pas celle de l'automate. Pour une graine de generateur, la
section 11 (p. 162) en tire la regle pratique : une longueur de cycle depassant `2^32` demande
`N = 49`, et `N = 127` suffit pour depasser `2^64`.

## 3. Test A -- distribution des frequences de blocs (Annexe A)

L'Annexe A (p. 166) decrit sept procedures. La premiere, **A. Block Frequency Distribution** :

> Each of the 2^n possible n-blocks should occur with equal frequency. (n = 8 is used.)

Le corps du texte (p. 158) donne la statistique, equation (10.1) :

$$\chi^2 = \sum \frac{(p_o - p_e)^2}{p_e}$$

avec `nu = 2^n` degres de liberte pour des blocs de longueur `n`. Ici `n = 8`, donc 256 classes et
`nu = 256` : la valeur publiee est la **probabilite qu'une sequence vraiment aleatoire depasse** le
chi-deux observe. Une valeur proche de 0 ou de 1 signale une deviation.

In [4]:
def test_A(bits):
    """Annexe A, test A : frequence des 256 octets ; renvoie (chi2, p_upper, n_octets)."""
    octets = bits_to_bytes(bits)
    counts = np.bincount(octets, minlength=256).astype(float)
    expected = len(octets) / 256.0
    chi2 = float(((counts - expected) ** 2 / expected).sum())
    return chi2, float(stats.chi2.sf(chi2, df=255)), len(octets)


# Controle de plomberie : sur des bits tires a pile ou face, p doit etre uniforme.
rng = np.random.default_rng(2026)
p_values = [test_A(rng.integers(0, 2, size=64 * 1024).astype(np.uint8))[1] for _ in range(20)]
print('temoin aleatoire -- p sur 20 tirages :', np.round(p_values, 2))
print('moyenne :', round(float(np.mean(p_values)), 3), '(attendu ~0,5 si le test est calibre)')
print('fraction sous 0,05 :', float(np.mean(np.array(p_values) < 0.05)), '(attendu ~0,05)')
print('fraction au-dessus de 0,95 :', float(np.mean(np.array(p_values) > 0.95)), '(attendu ~0,05)')

temoin aleatoire -- p sur 20 tirages : [0.79 0.04 0.14 0.43 0.01 0.73 0.73 0.46 0.98 0.09 0.51 0.12 0.89 0.48
 0.86 0.47 0.18 0.33 0.99 0.8 ]
moyenne : 0.502 (attendu ~0,5 si le test est calibre)
fraction sous 0,05 : 0.1 (attendu ~0,05)
fraction au-dessus de 0,95 : 0.1 (attendu ~0,05)


Le temoin ci-dessus est un **controle positif de l'instrument** : soumis a de vrais tirages, le test
doit rendre des `p` uniformes sur [0, 1]. Sans ce controle, un `p` extreme sur la regle 30 serait
indiscernable d'un test mal calibre.

Passons la regle 30 au test A, aux tailles et longueurs de la Table 10.1, et posons en regard les
valeurs que Wolfram publie (ligne A, colonnes 1 a 6).

In [5]:
PUBLISHED_A = [
    ('N=17, L=8k', 17, 8 * 1024, 0.0039),
    ('N=17, L=64k', 17, 64 * 1024, 1.0000),
    ('N=23, L=64k', 23, 64 * 1024, 0.0456),
    ('N=29, L=64k', 29, 64 * 1024, 0.7375),
    ('N=37, L=64k', 37, 64 * 1024, 0.3852),
    ('N=49, L=64k', 49, 64 * 1024, 0.8003),
]

print(f"{'cas':14s} {'chi2':>8s} {'p mesure':>10s} {'p Wolfram':>10s}  lecture")
for label, N, Lx, pub in PUBLISHED_A:
    chi2, p, nb = test_A(temporal_bits(30, N, Lx))
    flag = 'DEVIE' if (p < 0.05 or p > 0.95) else 'conforme'
    print(f'{label:14s} {chi2:8.2f} {p:10.4f} {pub:10.4f}  {flag}')

cas                chi2   p mesure  p Wolfram  lecture


N=17, L=8k       231.50     0.8519     0.0039  conforme


N=17, L=64k      422.50     0.0000     1.0000  DEVIE


N=23, L=64k      233.81     0.8253     0.0456  conforme


N=29, L=64k      269.44     0.2556     0.7375  conforme


N=37, L=64k      252.50     0.5325     0.3852  conforme


N=49, L=64k      226.44     0.9007     0.8003  conforme


**Lecture du resultat -- et ce qu'elle ne dit pas.** Le resultat se lit en deux temps, et il faut
resister au premier.

**Ce qui se reproduit** : le **mecanisme**. Le seul cas rejete est `N=17, L=64k` (`p = 0,0000`),
c'est-a-dire celui ou l'echantillon vaut **six fois la periode** du registre (65 536 pour 10 846) :
la sequence mesuree n'est plus un echantillon de dynamique, c'est la meme boucle repetee six fois.
Le meme `N=17` sur `L=8k` -- echantillon plus court que la periode -- n'est **pas** rejete
(`p = 0,8519`). Le phenomene n'est donc pas "petit `N` = mauvais", c'est "echantillon plus long que
la periode = artefact".

La ou la mise en garde de Wolfram est plus fine que mon propre enonce : `N=23` a une periode de
38 249 pour un echantillon de 65 536, soit **1,7 tour**, et le test ne rejette **pas** (`p = 0,8253`).
Le seuil n'est donc pas net -- c'est un ordre de grandeur, pas un critere. Wolfram parle de periode
"comparable" a la longueur echantillonnee, et cette mesure precise ce que *comparable* veut dire :
a 1,7 tours rien ne se voit, a 6 tours tout se voit. Pour `N >= 29` la periode depasse la fenetre de
recherche et le rapport tombe sous 1.

**Ce qui ne se reproduit pas** : les **valeurs exactes**. Pour `N=17, L=64k` Wolfram publie
`1,0000` quand on mesure `0,0000` ; pour `N=23` il publie `0,0456` quand on mesure `0,8253`. Les
deux mesures sont extremes dans le premier cas, mais **de part et d'autre de la distribution** :
les deux rejettent l'hypothese d'alea, l'une en haut, l'autre en bas. La cause n'est pas
mysterieuse : le protocole publie ne dit ni **quel site** on lit, ni **ou** se trouve le site non
nul, ni par quel bout on ordonne les bits dans l'octet. Ce sont trois degres de liberte qui changent
la sequence echantillonnee sans changer l'automate.

Une reproduction **fidele au chiffre** demanderait le code de Don Mitchell, que le texte ne donne
pas. Ce carnet ne peut donc pas pretendre valider la Table 10.1 ligne par ligne ; il peut valider
l'**enonce conditionnel** qui l'accompagne -- et c'est cet enonce qui porte la lecon d'usage.

## 4. Temoins negatifs -- ce que le meme test fait a des generateurs reputes

Un test qui ne rejette jamais rien ne mesure rien. La Table 10.2 (p. 160) fournit exactement les
temoins qu'il faut, et sa legende en donne tous les parametres :

- **LFSR** (linear feedback shift register) de longueur `N`, periode `2^N - 1`, **taps en positions
  14 et 17** pour `N = 17`, **27 et 29** pour `N = 29` ;
- **LCG** : `x' = (1103515245x + 12345) mod 2^31`, graine `x = 1` ("used, for example, in many
  implementations of the UNIX operating system") ;
- **CA60** : l'automate lineaire de l'equation (2.4), registre de taille `N = 29`.

Wolfram annonce le resultat avant de le mesurer (p. 159) : les LFSR, "while provably random in some
respects, are revealed as significantly nonrandom by several of the tests used here", et "many
sequences obtained from linear congruential generators are also found to be significantly
nonrandom". Le CA60 est le cas le plus net de sa table.

In [6]:
def lfsr_bits(n, taps, n_bits):
    """LFSR de longueur n, taps 1-indexes (Wolfram 1986, legende de la Table 10.2)."""
    state = [0] * n
    state[n // 2] = 1
    out = np.empty(n_bits, dtype=np.uint8)
    for t in range(n_bits):
        out[t] = state[0]
        fb = 0
        for tap in taps:
            fb ^= state[n - tap]
        state = state[1:] + [fb]
    return out


def lcg_bits(n_bits, seed=1):
    """LCG x' = (1103515245x + 12345) mod 2^31, bits de poids fort en tete."""
    M = 2 ** 31
    x, vals = seed, []
    for _ in range(n_bits // 31 + 2):
        x = (1103515245 * x + 12345) % M
        vals.append(x)
    bits = np.zeros(len(vals) * 31, dtype=np.uint8)
    for i, v in enumerate(vals):
        for b in range(31):
            bits[i * 31 + b] = (v >> (30 - b)) & 1
    return bits[:n_bits]


CONTROLES = [
    ('CA30  N=49', temporal_bits(30, 49, L)),
    ('CA60  N=29', temporal_bits(60, 29, L)),
    ('LFSR  n=17', lfsr_bits(17, (14, 17), L)),
    ('LFSR  n=29', lfsr_bits(29, (27, 29), L)),
    ('LCG  2^31', lcg_bits(L)),
]

print(f"{'generateur':12s} {'chi2':>9s} {'p (test A)':>12s}")
for label, seq in CONTROLES:
    chi2, p, _ = test_A(seq)
    print(f'{label:12s} {chi2:9.2f} {p:12.3g}')

generateur        chi2   p (test A)
CA30  N=49      226.44        0.901
CA60  N=29     1739.81    4.89e-219
LFSR  n=17      234.81        0.813
LFSR  n=29      750.44     2.77e-50
LCG  2^31       609.38     4.88e-31


**Lecture du resultat** : le test A **separe** la regle 30 de trois des quatre temoins de
construction **lineaire**. Le CA60 est rejete avec une probabilite astronomiquement faible
(`4,9e-219`), le LCG aussi (`4,9e-31`), et le LFSR de longueur 29 aussi (`2,8e-50`), quand la regle
30 reste au centre (`0,901`). C'est la revendication centrale de Wolfram 1986 : les schemes lineaires
sont "provably random in some respects" et pourtant detectables, parce que leur structure algebrique
laisse une signature.

**Deux nuances que la mesure impose, et qu'il faut ecrire.**

1. Le **LFSR de longueur 17 passe** le test A (`0,813`, il n'est pas rejete). Wolfram publie pour lui
   `0,0390`, une valeur basse mais elle aussi non concluante a ce seul test : le rejet des LFSR vient
   de la **batterie**, pas d'un test isole. Un carnet qui conclurait "le test A rejette les LFSR" a
   partir d'une seule ligne serait en avance sur sa propre mesure.
2. Un `p` eleve n'est pas une preuve d'alea : il dit seulement que ce test-la ne voit rien. La
   section 6 ajoute un test qui, lui, est **decisif** sur la difference de nature entre une sequence
   lineaire et la regle 30.

## 5. Test C -- blocs distincts (et un resultat negatif assume)

Deuxieme procedure de l'Annexe A (p. 166) :

> C. Distinct Blocks Distribution. The frequencies with which p out of q successive m-blocks are
> distinct should follow a definite distribution. (m = 4, q = 4 are used.)

On groupe les bits en blocs de 4, on prend les groupes de 4 blocs successifs, on compte combien de
valeurs **distinctes** chacun contient, et on confronte la distribution observee a la distribution
exacte d'un tirage avec remise dans un alphabet de 16 symboles.

La distribution attendue se calcule exactement : la probabilite que `p` valeurs distinctes
apparaissent en `q` tirages vaut `(16)_p * S(q,p) / 16^q`, ou `(16)_p` est la factorielle
descendante et `S` le nombre de Stirling de seconde espece.

In [7]:
from math import comb, factorial


def test_C(bits, m=4, q=4):
    """Annexe A, test C : nombre de blocs distincts parmi q blocs m-bits successifs."""
    nm = len(bits) // m
    vals = bits[:nm * m].reshape(nm, m).dot(1 << np.arange(m - 1, -1, -1))
    ng = len(vals) // q
    groups = vals[:ng * q].reshape(ng, q)
    obs = np.bincount([len(set(g)) for g in groups], minlength=q + 1)[1:].astype(float)

    space = 2 ** m
    exp = np.zeros(q)
    for p in range(1, q + 1):
        fall = 1.0
        for k in range(p):
            fall *= space - k
        stirling = sum((-1) ** j * comb(p, j) * (p - j) ** q for j in range(p + 1)) // factorial(p)
        exp[p - 1] = fall * stirling / space ** q
    expected = exp * ng
    chi2 = float(((obs - expected) ** 2 / expected).sum())
    return chi2, float(stats.chi2.sf(chi2, df=q - 1)), obs.astype(int).tolist()


for label, seq in CONTROLES:
    chi2, p, obs = test_C(seq)
    print(f'{label:12s} chi2={chi2:7.2f}  p={p:.4f}  effectifs (1,2,3,4 blocs distincts)={obs}')

CA30  N=49   chi2=   1.13  p=0.7708  effectifs (1,2,3,4 blocs distincts)=[2, 103, 1269, 2722]
CA60  N=29   chi2= 303.58  p=0.0000  effectifs (1,2,3,4 blocs distincts)=[6, 242, 1501, 2347]
LFSR  n=17   chi2=   3.14  p=0.3712  effectifs (1,2,3,4 blocs distincts)=[1, 88, 1249, 2758]
LFSR  n=29   chi2= 145.37  p=0.0000  effectifs (1,2,3,4 blocs distincts)=[11, 170, 1293, 2622]
LCG  2^31    chi2=   3.83  p=0.2804  effectifs (1,2,3,4 blocs distincts)=[0, 108, 1308, 2680]


**Lecture du resultat -- un test qui discrimine, mais pas les memes que le test A.** Le test C ne
rejette pas "les generateurs lineaires" en bloc : il rejette le **CA60** (`p = 0,0000`) et le
**LFSR de longueur 29** (`p = 0,0000`), et il laisse passer le **LFSR de longueur 17** (`p = 0,3712`)
et le **LCG** (`p = 0,2804`). La regle 30 passe (`p = 0,7708`).

Ce n'est pas une redite du test A : les deux tests ne voient pas les memes cas.

| Generateur | Test A | Test C | Rejete au moins une fois |
|---|---|---|---|
| CA30 (N=49) | passe (0,901) | passe (0,771) | **non** |
| CA60 (N=29) | rejete (4,9e-219) | rejete (0,0000) | oui |
| LFSR n=17 | passe (0,813) | passe (0,371) | **non** |
| LFSR n=29 | rejete (2,8e-50) | rejete (0,0000) | oui |
| LCG 2^31 | rejete (4,9e-31) | passe (0,280) | oui |

**Le cas qui compte est la ligne du LFSR de longueur 17** : il **echappe aux deux tests**. Un carnet
qui n'en aurait mesure qu'un, ou qui aurait conclu "les generateurs lineaires sont tous detectes",
serait passe a cote. Ce generateur est lineaire, sa sequence est par construction entierement
contenue dans 17 bits (section 6 le montrera), et pourtant ni la frequence des octets ni celle des
blocs distincts ne le voient sur 65 536 bits.

C'est exactement pourquoi l'Annexe A compte **sept** procedures et non deux. La these de Wolfram
(p. 159) dit "several of the tests used here", pas *all of them*, et sa propre Table 10.2 donne pour
ce generateur `0,0390` au test A -- une valeur basse, elle non plus non concluante. La mesure
reproduit donc aussi ce detail-la.

La mise en garde de l'Annexe A (p. 166) prend ici tout son sens :

> If the confidence interval is very close to zero or one, then the observed chi-deux is unlikely
> to be produced from a random sequence [...] Of course, if say a total of k tests are done, it is
> to be expected that the confidence interval for at least one of them will be less than 1/k.

Un resultat isole ne conclut pas, et **deux** non plus. Ce carnet en mesure deux sur sept, et le dit.

## 6. Complexite lineaire -- le test qui tranche sur la nature

Les tests A et C mesurent des **frequences**. Aucun des deux ne dit pourquoi les generateurs
lineaires doivent echouer quelque part : ils le constatent. La raison est structurelle, et elle se
mesure directement par la **complexite lineaire** : la longueur du plus court registre a decalage
theorique capable de reproduire la sequence (algorithme de Berlekamp-Massey).

C'est exactement l'argument que Wolfram oppose aux methodes lineaires (p. 162-163) :

> The primary feature of linear schemes is that they can be analysed by algebraic methods. As a
> consequence, certain randomness properties can be proved for the sequences they generate [...] But
> the simplicity in structure which underlies this analysis also limits the degree of randomness
> that such schemes can produce.

Une sequence de complexite lineaire `c` est **entierement determinee** par `2c` bits. Pour un LFSR de
longueur `n`, `c = n` : 17 ou 29 bits suffisent a reconstruire la sequence. Pour un generateur qui
n'est pas lineaire, `c` doit croitre avec la longueur echantillonnee, et se stabiliser autour de
`L/2` -- la valeur d'une sequence aleatoire.

In [8]:
def linear_complexity(bits):
    """Longueur du plus court LFSR engendrant la sequence (Berlekamp-Massey, sur GF(2))."""
    s = [int(b) for b in bits]
    c, b = [1], [1]
    length, m = 0, 1
    for i in range(len(s)):
        d = s[i]
        for j in range(1, length + 1):
            if j < len(c):
                d ^= c[j] & s[i - j]
        if d == 1:
            prev = c[:]
            shifted = [0] * m + b
            if len(shifted) > len(c):
                c += [0] * (len(shifted) - len(c))
            for k in range(len(shifted)):
                c[k] ^= shifted[k]
            if 2 * length <= i:
                length, b, m = i + 1 - length, prev, 1
            else:
                m += 1
        else:
            m += 1
    return length


N_BITS = 4096
print(f"{'generateur':12s} {'complexite lineaire':>20s}  (L/2 = {N_BITS // 2})")
for label, seq in [
    ('CA30  N=49', temporal_bits(30, 49, N_BITS)),
    ('CA110 N=49', temporal_bits(110, 49, N_BITS)),
    ('LFSR  n=17', lfsr_bits(17, (14, 17), N_BITS)),
    ('LFSR  n=29', lfsr_bits(29, (27, 29), N_BITS)),
    ('LCG  2^31', lcg_bits(N_BITS)),
]:
    print(f'{label:12s} {linear_complexity(seq):20d}')

generateur    complexite lineaire  (L/2 = 2048)


CA30  N=49                   2049


CA110 N=49                   1674
LFSR  n=17                     17
LFSR  n=29                     29


LCG  2^31                    2051


**Lecture du resultat** : la separation est nette et elle est **de nature**, pas de degre. Les deux
LFSR ont une complexite lineaire egale a leur longueur (17 et 29) : toute leur sequence tient dans
une poignee de bits. La regle 30 mesuree sur 4 096 bits atteint `L/2`, la valeur que prend une
sequence aleatoire. Wolfram ne publie pas cette mesure dans l'article de 1986, mais il l'enonce en
prose : la simplicite algebrique des methodes lineaires **borne** ce qu'elles peuvent produire.

Une remarque d'honnetete, qui limite la portee de cette section : le LCG mesure ici une complexite
lineaire elevee (`L/2`), alors que le test A le rejetait. Les deux resultats ne se contredisent pas --
la complexite lineaire elevee d'un LCG est un fait connu, du a l'arithmetique modulaire -- mais ils
montrent que **chaque test voit une propriete differente**. C'est precisement pourquoi la batterie
de l'Annexe A en compte sept.

## 7. La reception -- ce que Gray oppose a la these

Wolfram ne se contente pas de proposer un generateur : il en fait un argument. Gray (2003, p. 203)
resume l'usage reel et pose la limite dans le meme paragraphe :

> the sequence of colors attained by the cell at the origin forms the basis for the random number
> generator used by Mathematica. This apparently perfectly random behavior was first noticed by
> Wolfram in 1984, and it seems to defy any sort of rigorous analysis. The best mathematical result
> that I know about Rule 30 is due to Erica Jen.

La these de Wolfram est **empirique** : la sequence passe les tests, donc elle est aleatoire "for
practical purposes" -- la formule exacte de la section 10 (p. 161) :

> The fact that deviations from randomness have not been found by any of the statistical procedures
> considered lends strong support to the belief that sequences produced by Eq. (3.1) with large N
> are indeed random for practical purposes.

Gray ne conteste pas les mesures ; il conteste le glissement de "passe les tests" a "est
aleatoire", et il le fait en deux endroits. Sur la regle 30 d'abord : il note que Wolfram croit
aussi la regle 30 **universelle**, et tranche -- "I find this to be sheer wishful thinking" (p. 203).
Sur la methode ensuite, en reprenant l'argument que Wolfram emploie contre les autres (p. 203) :
"unless a CA is obviously simple [...] it can simulate a universal Turing machine" est une these
generale que Gray juge non etayee.

Ce carnet mesure exactement ce que la these empirique permet de mesurer, et rien de plus :
**absence de deviation detectable par deux tests sur sept**, dans un regime ou la periode du
registre depasse l'echantillon. L'absence de detection n'est pas une preuve d'alea, et la section
10 de Wolfram le dit elle-meme (p. 158) : "one can never definitively conclude by studying a finite
sample that the complete sequence is not random".

## 8. La regle 110 -- la structure qui porte l'universalite

L'universalite de la regle 110 est un **theoreme**, pas une observation. Cook 2004 (p. 1) :

> The purpose of this paper is to prove a conjecture made by Stephen Wolfram in 1985, that an
> elementary one dimensional cellular automaton known as "Rule 110" is capable of universal
> computation.

Ce carnet ne re-prouve pas ce theoreme, et il ne peut pas le tester : l'universalite est une
propriete de la **famille** des configurations initiales, pas d'une trajectoire. Ce qu'il peut
mesurer, c'est la structure que la construction de Cook exploite -- et qui distingue la regle 110 de
la regle 30.

Cook (2004, section 2.3, p. 5) decrit les systemes de gliders comme le materiau de sa preuve :

> Each moving point is called a glider. [...] When two gliders meet, a lookup table specifies what
> the result of the collision should be.

La regle 110 possede un **fond periodique** (l'ether) sur lequel se deplacent des gliders. C'est ce
fond qui rend le codage possible : il fournit une memoire stable, ce qu'un regime chaotique ne
fournit pas. L'organe classe deja les deux regles ; mesurons la structure.

In [9]:
def spacetime(rule, n_cells, n_steps, seed_site=None):
    """Diagramme espace-temps : une ligne par pas de temps."""
    state = seed_single_nonzero(n_cells, seed_site)
    rows = np.empty((n_steps, n_cells), dtype=np.uint8)
    for t in range(n_steps):
        rows[t] = state
        state = wolfram_step(state, rule)
    return rows


for r in (30, 110):
    c = classify_rule(r, n_cells=64, n_steps=500)
    print(f'regle {r:3d} : {c["class_label"]:22s} densite={c["indicators"]["asymptotic_density"]:.4f} '
          f'ecart-type={c["indicators"]["asymptotic_std"]:.4f}')

pat = spacetime(110, 600, 500)
print()
print('regle 110, extrait espace-temps (x de 250 a 400, un pas sur quatre) :')
for t in range(440, 500, 4):
    print(''.join('#' if v else '.' for v in pat[t, 250:400]))

regle  30 : III (known+measured)   densite=0.4988 ecart-type=0.0579
regle 110 : IV (known+measured)    densite=0.5733 ecart-type=0.0399

regle 110, extrait espace-temps (x de 250 a 400, un pas sur quatre) :
#...#..##.#####...#..##.#####...#..##.#####...#..##...................................................................................................
..#..##.#####...#..##.#####...#..##.#####...#..##.#...................................................................................................
#..##.#####...#..##.#####...#..##.#####...#..##.###...................................................................................................
.##.#####...#..##.#####...#..##.#####...#..##.#####...................................................................................................
#.#####...#..##.#####...#..##.#####...#..##.#####.#...................................................................................................
#####...#..##.#####...#..##.#####...#.

In [10]:
def spacetime_period(pat, a, b, t0, t1, max_dt=14, max_dx=20):
    """Plus petites periodes (dt, dx) telles que pat[t+dt, x+dx] == pat[t, x] sur la fenetre."""
    found = []
    for dt in range(1, max_dt + 1):
        for dx in range(-max_dx, max_dx + 1):
            if all(np.array_equal(pat[t + dt, a:b], pat[t, a - dx:b - dx]) for t in range(t0, t1)):
                found.append((dt, dx))
    return found


def spatial_period(pat, a, b, t0, t1, max_shift=40):
    """Plus petit decalage s tel que pat[t, x] == pat[t, x + s] sur toute la fenetre."""
    for s in range(1, max_shift + 1):
        if all(np.array_equal(pat[t, a:b - s], pat[t, a + s:b]) for t in range(t0, t1)):
            return s
    return None


periodes_ether = spacetime_period(pat, 250, 380, 420, 480)
print('periodes espace-temps de l ether (fenetre x=250..380, t=420..480) :', periodes_ether)
print('periode spatiale de l ether sur la meme fenetre :',
      spatial_period(pat, 250, 380, 420, 480))

# Controle : la meme mesure sur la regle 30 ne doit rien trouver
pat30 = spacetime(30, 600, 500)
print('meme mesure sur la regle 30 (temoin negatif) :',
      spacetime_period(pat30, 250, 380, 420, 480), '/',
      spatial_period(pat30, 250, 380, 420, 480))

# Contraste des regimes : persistance de la structure
for r in (30, 110):
    p = spacetime(r, 300, 400)
    dens = p.mean(axis=1)
    print(f'regle {r:3d} : densite moyenne (100 derniers pas)={dens[-100:].mean():.3f} '
          f'ecart-type={dens[-100:].std():.4f}')

periodes espace-temps de l ether (fenetre x=250..380, t=420..480) : [(7, 0), (14, 0)]
periode spatiale de l ether sur la meme fenetre : None
meme mesure sur la regle 30 (temoin negatif) : [] / None


regle  30 : densite moyenne (100 derniers pas)=0.501 ecart-type=0.0262
regle 110 : densite moyenne (100 derniers pas)=0.582 ecart-type=0.0112


**Lecture du resultat** : la fenetre choisie est invariante par les translations espace-temps
`(dt = 7, dx = 0)` et `(dt = 14, dx = 0)` -- le fond se repete **tous les 7 pas de temps**, a
position spatiale fixe. Le controle negatif est net : la meme mesure sur la regle 30 ne trouve
**aucune** translation, ni en temps ni en espace.

**Ce que la mesure ne dit pas, et qu'il faut ecrire.** La periode **spatiale** n'est pas etablie :
la fonction `spatial_period` ne trouve aucun decalage invariant jusqu'a sa borne de recherche `max_shift = 40`, alors que la
litterature sur la regle 110 attribue au fond une periode spatiale de 14. Deux lectures sont
possibles et la mesure seule ne tranche pas : soit la fenetre `x = 250..380` contient autre chose
que du fond pur (un defaut, une frontiere de phase), soit l'invariance spatiale n'est pas
verifiable sur une bande de temps aussi courte dans un systeme ou des particules se deplacent.

Ce desaccord est instructif, et c'est pour cela qu'il est garde : une periode **espace-temps**
`(dt, dx)` est ce dont la construction de Cook a besoin -- c'est un repere qui se deplace, pas une
grille fixe. Chercher une periode purement spatiale revient a supposer que la structure est
immobile, ce qui est exactement l'hypothese que les gliders invalident.

C'est la difference que Cook exploite : un fond periodique est une **memoire**, une horloge. Un
glider qui s'y deplace peut coder un symbole, deux gliders qui se rencontrent peuvent implementer une
transition. Sur la regle 30, il n'y a rien a coder : le motif se defait, et la mesure le montre
aussi -- aucune translation, dans aucune direction.

La densite apporte un indice complementaire : la regle 110 garde une densite plus elevee et plus
stable que la regle 30 (`0,582` contre `0,501`, ecart-type `0,0112` contre `0,0262`). Mais il ne
faut pas surinterpreter ce nombre -- il mesure un remplissage, pas une structure. Le diagramme et
les periodes espace-temps sont les preuves ; la densite n'est qu'un indice.

## 9. Ce que le theoreme de Cook etablit -- et ce qu'il coute

Cook 2004 construit l'universalite en deux etages, et il les nomme dans son plan :

1. **section 2.2 (p. 4), les systemes de tags cycliques** : un systeme de reecriture sur bande finie,
   dont Cook montre qu'il est universel en simulant un systeme de tags ordinaire ;
2. **section 2.3 (p. 5), les systemes de gliders** : des particules mobiles dont les collisions
   obeissent a une table de correspondance -- et la regle 110 en fournit une implementation.

La conclusion (p. 1) est plus forte que "on peut calculer avec la regle 110" :

> implying that many questions concerning its behavior, such as whether a particular sequence of
> bits will occur, or whether the behavior will become periodic, are formally undecidable.

C'est le sens du mot *universel* ici : la regle 110 encode un modele de calcul, donc elle **herite
son indecidabilite**. Aucun test statistique ne peut etablir cela -- l'indecidabilite est un resultat
de theorie de la calculabilite, et le nier demande une preuve, pas une mesure.

Wolfram 1984 avait formule la conjecture au niveau des **classes**, avant la preuve (resume, p. 1) :

> The fourth class is probably capable of universal computation, so that properties of its infinite
> time evolution may be undecidable.

La difference de statut entre 1984 et 2004 est exactement celle que ce carnet met en scene : en
1984, la classe IV "est probablement" capable de calcul universel ; en 2004, une regle precise de
cette classe est **prouvee** universelle. Entre les deux, il n'y a pas plus de mesures -- il y a une
construction.

## 10. La reception de la regle 110 -- priorite, accord de non-divulgation, action juridique

Gray (2003, p. 203) raconte la chronologie, et son recit est le document de reception le plus precis
dont on dispose :

> The universal behavior of Rule 110 was first conjectured by Wolfram around 1985. The result was
> proved by Matthew Cook in the mid to late 1990s. However, Cook was prevented from publishing this
> result, because he had signed an agreement, as an employee of Wolfram Research, not to disclose
> his proof until ANKS was published. Because the appearance of ANKS was long delayed, Cook tested
> this agreement by presenting his result at a conference at the Santa Fe Institute, at which I was
> present. His talk was the highlight of the conference, but Wolfram threatened legal action,
> preventing it from appearing in the conference proceedings. Now that the nondisclosure agreement
> has expired, Cook's proof should become available.

Et son verdict sur la preuve elle-meme, qui n'est pas de complaisance :

> I have seen it and I highly recommend it, because it shows just how tenuous the original
> conjecture was -- the proof just barely works.

Deux lecons de reception, qui ne sont pas des anecdotes :

- **Un resultat peut etre vrai et retenu.** La priorite de la conjecture est a Wolfram (1985), la
  preuve a Cook (milieu-fin des annees 1990), la publication a 2004. Un lecteur qui citerait "Cook,
  universality de la regle 110" sans la date croirait a une decouverte publique de 1995.
- **La preuve est etroite.** "The proof just barely works" n'est pas une condamnation : c'est
  l'information la plus utile du paragraphe. Une universalite qui tiendrait avec une marge
  confortable serait une autre these ; celle qui tient de justesse dit quelque chose du regime de la
  classe IV.

## 11. Synthese -- quatre statuts, et ce que ce carnet a reellement etabli

| Affirmation | Source | Statut | Mesure ici |
|---|---|---|---|
| La regle 30 produit des sequences sans deviation detectable en regime non periodique | Wolfram 1986, p. 159-161 | these empirique | **reproduite** (test A et test C, `N >= 29`) |
| Les deviations apparaissent quand la periode du registre approche la longueur echantillonnee | Wolfram 1986, p. 159 | enonce conditionnel | **reproduite** et chiffree (periodes mesurees ; seuil a 1,7 tour non detecte, a 6 tours detecte) |
| Les LFSR et les LCG sont detectes par la batterie | Wolfram 1986, p. 159-160 | these empirique | **reproduite en partie** (A et C rejettent CA60, LFSR n=29 et LCG ; le LFSR n=17 echappe aux deux) |
| Les generateurs lineaires ont une structure identifiable | Wolfram 1986, p. 162-163 | argument structurel | **reproduite** (complexite lineaire : 17 et 29 pour les LFSR, `L/2` pour la regle 30) |
| Les valeurs exactes de la Table 10.1 sont reproductibles | Wolfram 1986, p. 159 | -- | **non reproduites** (protocole sous-specifie : site, position de graine, ordre des bits) |
| La regle 110 est universelle | Cook 2004, p. 1 | **theoreme** | non mesurable -- cite, non teste |
| La classe IV est "probablement" universelle | Wolfram 1984, resume | conjecture | encadree par le theoreme de 2004 |
| La regle 30 elle-meme serait universelle | Gray 2003, p. 203 | opinion de Wolfram, contestee | non mesuree -- "sheer wishful thinking" selon Gray |

La colonne qui compte est la troisieme. Un carnet qui traiterait les huit lignes de la meme facon
serait faux sur au moins cinq.

## Exercice 1 -- Sensibilite du protocole : le site echantillonne

Le carnet a fixe `site=0` et une graine au centre du registre, parce que le texte de 1986 ne les
precise pas. Cet exercice mesure l'effet de ce choix.

**Objectif** : pour `N = 17` et `L = 64k`, calculer le `p` du test A pour trois sites (0, `N//4`,
`N//2`) et trois positions de graine (0, `N//2`, `N - 1`). Remplir un tableau 3x3 et dire si le
verdict (rejet ou non) change.

**Indice** : `temporal_bits(30, N, L, site=s, seed_site=g)` accepte les deux parametres.

**Question** : si le verdict change selon le site, que peut-on encore affirmer sur "la regle 30 a
N=17" ?

In [11]:
N, Lx = 17, 64 * 1024
sites = [0, N // 4, N // 2]
graines = [0, N // 2, N - 1]

# TODO etudiant : remplir la grille des p du test A
resultat = None  # TODO

# Etape 1 : boucler sur les sites et les graines
# Etape 2 : appeler temporal_bits(30, N, Lx, site=s, seed_site=g) puis test_A
# Etape 3 : afficher le verdict (rejet si p < 0.05 ou p > 0.95) pour chaque case
print('Exercice a completer : grille de sensibilite du protocole')

Exercice a completer : grille de sensibilite du protocole


## Exercice 2 -- La regle 90 est-elle lineaire, et le voit-on ?

La regle 90 (XOR du voisin gauche et du voisin droit) engendre le triangle de Sierpinski : un motif
visuellement spectaculaire, mais produit par une operation **lineaire**.

**Objectif** : mesurer la complexite lineaire de la regle 90 sur 4 096 bits, et la comparer a celle
de la regle 30 et a celle d'un LFSR.

**Indice** : `linear_complexity(temporal_bits(90, 49, 4096))`.

**Question** : si la regle 90 a une complexite lineaire basse, que cela implique-t-il pour un
carnet qui presenterait son triangle comme une preuve de "complexite" ?

In [12]:
lc_90 = None  # TODO : complexite lineaire de la regle 90
lc_30 = None  # TODO : complexite lineaire de la regle 30 (reference)

# Etape 1 : extraire 4096 bits de la regle 90 avec temporal_bits
# Etape 2 : passer la sequence a linear_complexity
# Etape 3 : comparer aux valeurs de reference de la section 6
print('Exercice a completer : complexite lineaire de la regle 90')

Exercice a completer : complexite lineaire de la regle 90


## Exercice 3 -- Mesurer la periode de l'ether de la regle 110 sur une autre fenetre

La section 8 a mesure les translations espace-temps `(dt = 7, dx = 0)` et `(dt = 14, dx = 0)` sur la
fenetre `x = 250..380`, et n'a trouve **aucune** invariance purement spatiale. Cet exercice verifie
que ces deux resultats ne sont pas des artefacts de la fenetre choisie.

**Objectif** : refaire la mesure sur deux autres fenetres (par exemple `x = 200..320` et
`x = 300..420`), puis sur un diagramme plus large (`n_cells = 800`, en gardant `n_steps = 500`).

**Indice** : `spacetime(110, 800, 500)` puis `spacetime_period(...)` et `spatial_period(...)`.

**Question** : si la periode temporelle reste 7 sur toutes les fenetres alors que l'invariance
spatiale ne se trouve nulle part, que mesure-t-on reellement -- le fond de l'automate, ou le fait que
les particules qui s'y deplacent cassent toute periodicite spatiale ?

In [13]:
results = None  # TODO : liste des periodes trouvees par fenetre

# Etape 1 : construire un diagramme plus large avec spacetime(110, 800, 500)
# Etape 2 : appliquer spacetime_period sur deux fenetres disjointes
# Etape 3 : comparer avec (7, 0) et commenter
print('Exercice a completer : robustesse de la periode de l ether')

Exercice a completer : robustesse de la periode de l ether


## Conclusion -- ce que la serie garde de ce pli

Ce carnet ferme les deux points de la definition du pli 2 que la serie ne couvrait pas, et il les
ferme avec des statuts differents, ce qui est le fond du sujet :

1. **La regle 30 comme generateur** est une these **empirique**. Elle se mesure, et la mesure
   reproduit ce qui est reproductible : l'absence de deviation detectable quand la periode du
   registre depasse l'echantillon, et le rejet des generateurs lineaires par une batterie de tests
   -- avec la complexite lineaire comme discriminant de nature. Elle ne reproduit pas les valeurs
   exactes de la Table 10.1, et le carnet dit pourquoi plutot que de choisir un jeu de parametres
   qui tombe juste.
2. **L'universalite de la regle 110** est un **theoreme**. Il ne se mesure pas. Ce qui se mesure,
   c'est la structure qui le rend possible -- le fond periodique de periode 7 en temps, 14 en
   espace -- et cette mesure explique pourquoi la classe IV peut coder la ou la classe III ne le
   peut pas.

**Limites assumees** : deux tests sur les sept de l'Annexe A ; un seul site echantillonne ; pas de
reproduction des periodes de la Table 9.2 au-dela de `N = 23` (fenetre de recherche bornee a
300 000 pas) ; la preuve de Cook n'est ni lue integralement ni verifiee, seulement citee.

**References** (copies archivees, `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\`) :

- Wolfram, S. (1986). *Random Sequence Generation by Cellular Automata*. Advances in Applied
  Mathematics 7, 123-169. Section 10 (p. 158-161), section 11 (p. 161-163), Annexe A (p. 166-167).
- Wolfram, S. (1984). *Universality and Complexity in Cellular Automata*. Physica D 10, 35-57.
  Resume (p. 1), definition des quatre classes (p. 5).
- Cook, M. (2004). *Universality in Elementary Cellular Automata*. Complex Systems 15(1), 1-40.
  Section 1 (p. 1-2), sections 2.2-2.3 (p. 4-5).
- Gray, J. (2003). *A Mathematician Looks at Wolfram's New Kind of Science*. Notices of the AMS
  50(2), 200-211. Regle 30 et generateur de Mathematica (p. 203), priorite de la regle 110 et
  accord de non-divulgation (p. 203).